# Notebook 8 - End-to-End Demonstration and Evaluation

Roadmap role: execute the fixed compound scenario through the complete Shepherd-AI software-simulation pipeline. Literature-driven rule: distinguish module format validity from actual scenario success. This notebook currently begins with a typed preflight; it must not be presented as complete until exact-scenario ASR and mission-assigned imagery are stored.

In [ ]:
from pathlib import Path
import os
import subprocess

IN_COLAB = bool(os.environ.get('COLAB_RELEASE_TAG'))
REPO = Path('/content/shepherd-ai') if IN_COLAB else Path.cwd()
if IN_COLAB and not (REPO / 'pyproject.toml').exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'codex/week8-end-to-end',
        'https://github.com/cyberuniversal/shepherd-ai.git', str(REPO),
    ], check=True)
if not (REPO / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook from the Shepherd-AI repository or clone it to /content/shepherd-ai.')
%cd {REPO}
%pip install -q -e .


## Fixed roadmap scenario preflight

This run stores an expected negative result while the map grounds `east` and `irrigation` to different objects. Do not bypass the clarification gate.

In [ ]:
!python scripts/run_week8_preflight.py \
  --output outputs/evaluations/week8_roadmap_scenario_typed_preflight.json


In [ ]:
import json
result_path = Path('outputs/evaluations/week8_roadmap_scenario_typed_preflight.json')
payload = json.loads(result_path.read_text(encoding='utf-8'))
summary = {
    'status': payload['pipeline_result']['status'],
    'clauses': len(payload['pipeline_result']['decomposition']['clauses']),
    'requested_drones': payload['pipeline_result']['decomposition']['total_requested_drones'],
    'blocking_clauses': payload['pipeline_result']['blocking_clauses'],
    'issues': payload['pipeline_result']['issues'],
}
summary


## Operator-resolved three-drone simulation

The operator selected East Field as the third drone's destination while retaining irrigation as the semantic inspection target. The simulator uses static formation slots, deterministic 2D interpolation, raw telemetry, and the configured runtime separation check. It is not a physical-flight or active-collision-avoidance model.

In [ ]:
!python scripts/run_week8_simulation.py \
  --output outputs/evaluations/week8_roadmap_scenario_simulation.json \
  --telemetry-output outputs/evaluations/week8_roadmap_scenario_telemetry.jsonl \
  --map-output outputs/visualizations/week8_roadmap_scenario_simulation.html


In [ ]:
from IPython.display import IFrame, display
simulation_payload = json.loads(
    Path('outputs/evaluations/week8_roadmap_scenario_simulation.json').read_text(encoding='utf-8')
)
display({key: simulation_payload['simulation'][key] for key in (
    'status', 'assignment_count', 'planned_makespan_min',
    'minimum_observed_separation_m', 'telemetry_records'
)})
IFrame('outputs/visualizations/week8_roadmap_scenario_simulation.html', width='100%', height=650)
